# Module 4: Agent as a Tool

Apply **Pattern 5**: wrap specialist agents as tools, so an orchestrator agent can delegate like a manager to experts. In Module 3 graph edges fixed the order of work; here the model decides it.

**When to use this pattern:**
- Clear hierarchy: one coordinator, many experts
- Each specialist needs its own tools and prompt
- The order of work depends on the request, so you let the model decide it
- Add / remove specialists without touching the orchestrator

**Key Strands primitives:** an `Agent` wrapped in a `@tool` function, and `Agent.as_tool()`.
The **docstring / description** is the routing signal: the orchestrator reads it to decide when and how to call each specialist.

**Prerequisites:** Modules 1–3. This module reuses Module 2's tools and the shared decision brief.

## Components in This Module

| Component | Type | What it does |
|-----------|------|-------------|
| `research_agent` | `@tool` wrapping an `Agent` | Gathers company metrics, benchmarks, competitive intelligence (has Module 2's tools) |
| `finance_agent` | `@tool` wrapping an `Agent` | Analyzes financial viability, ROI, unit economics |
| `legal_agent` | `@tool` wrapping an `Agent` | Reviews compliance risks and legal considerations |
| `writer_agent` | `Agent.as_tool(delegate=True)` | Writes the final investment memo, returned to the user as-is |
| `orchestrator` | `Agent(tools=[...])` | Coordinates the 4 specialists; the LLM decides routing |

In [ ]:
%pip install -r requirements.txt

In [ ]:
# ── Model ────────────────────────────────────────────────────────────────
# No model= is passed, so every Agent uses the Strands default on Amazon Bedrock:
# Claude Sonnet 4.6 through the global inference profile (global.anthropic.claude-sonnet-4-6).
# To use another model enabled in your account, create one and pass model=model to each Agent:
#   from strands.models import BedrockModel
#   model = BedrockModel(model_id="us.anthropic.claude-haiku-4-5-20251001-v1:0")  # faster
#   model = BedrockModel(model_id="amazon.nova-pro-v1:0")                        # Amazon Nova

import os, sys, time, json
sys.path.insert(0, os.path.join(os.getcwd(), "..", "02-single-agent"))
sys.path.insert(0, os.path.join(os.getcwd(), "..", "shared"))

from strands import Agent, tool
from decision_brief_tools import get_company_data, get_market_benchmarks, get_competitor_data
from decision_brief import DECISION_BRIEF

---

## Part 1: System Prompts

In Module 2 one prompt asked a single agent to research, analyze and write. Here each specialist gets a **narrow, focused system prompt**: it does one job and nothing else. The orchestrator's prompt describes the process, not the content.

In [ ]:
RESEARCH_PROMPT = (
    "You are a market research specialist. Use your tools to gather company data, "
    "industry benchmarks, and competitive intelligence. Return structured findings: data only."
)

FINANCE_PROMPT = (
    "You are a financial analyst. Analyze the investment brief and market data provided. "
    "Return: revenue projections, unit economics (CAC, LTV, payback period), "
    "ROI estimate, capital efficiency, and a financial verdict (Invest / Invest with conditions / Pass). "
    "Be specific with numbers. 200 words max."
)

LEGAL_PROMPT = (
    "You are a legal and compliance reviewer. Review the investment brief for: "
    "regulatory risks, data privacy concerns (GDPR, CCPA), contractual obligations, "
    "IP considerations, and any red flags for due diligence. "
    "Return a bullet-point list of legal risks with severity (High/Medium/Low). 150 words max."
)

WRITER_PROMPT = (
    "You are an investment memo writer. You receive the brief, research findings, financial "
    "analysis, and legal review. Produce a professional investment analysis memo:\n"
    "## Executive Summary (recommendation in one sentence)\n"
    "## Market Opportunity (size, growth, competitive position)\n"
    "## Financial Highlights (key metrics, ROI, projections)\n"
    "## Risk Assessment (top 3 risks with mitigations)\n"
    "## Recommendation (Invest / Invest with conditions / Pass, terms, conditions)\n"
    "Under 450 words. Be direct."
)

ORCHESTRATOR_PROMPT = (
    "You are an investment committee coordinator. For each investment request:\n"
    "1. Call research_agent to gather company and market data.\n"
    "2. Call finance_agent with the brief and research findings to get financial analysis.\n"
    "3. Call legal_agent with the brief to identify legal and compliance risks.\n"
    "4. Call writer_agent LAST, on its own, with one input that contains the brief, the research "
    "findings, the financial analysis, and the legal review. Its memo goes to the user as-is.\n"
    "For a follow-up question you can answer from the conversation so far, answer directly "
    "without calling the specialists again."
)

---

## Part 2: Wrap Specialists as Tools

**Three ways to use agents as tools in Strands:**

```python
# Option A: @tool function that calls an Agent (most control, multi-parameter)
@tool
def finance_agent(brief: str, research_context: str) -> str: ...

# Option B: pass an Agent directly in tools[] (simplest; one natural-language input)
orchestrator = Agent(tools=[researcher, analyst, writer])

# Option C: .as_tool() (custom name/description, preserve_context, delegate)
orchestrator = Agent(tools=[writer.as_tool(description="...", delegate=True)])
```

This module uses two of them:
- **Option A** for research, finance and legal: multi-parameter tools let the orchestrator pass precise arguments (the brief and the research context as separate fields).
- **Option C** for the writer, with `delegate=True`: the writer's memo becomes the orchestrator's final answer unchanged, so the orchestrator does not spend another model call restating it. A delegated tool must be called alone in its turn, which suits a "call this LAST" step.

Every specialist has `callback_handler=None`, so only the orchestrator prints. With delegation the writer's memo streams through the orchestrator once.

In [ ]:
@tool
def research_agent(topic: str) -> str:
    """Gather market data, company metrics, and competitive intelligence for an investment topic.

    Args:
        topic: The company or investment topic to research
    """
    worker = Agent(
        tools=[get_company_data, get_market_benchmarks, get_competitor_data],
        system_prompt=RESEARCH_PROMPT,
        callback_handler=None,
    )
    return str(worker(topic))


@tool
def finance_agent(brief: str, research_context: str) -> str:
    """Analyze financial viability: ROI, unit economics, projections, and investment verdict.

    Args:
        brief: The original investment brief
        research_context: Market and company data from research_agent
    """
    worker = Agent(system_prompt=FINANCE_PROMPT, callback_handler=None)
    return str(worker(f"Investment brief:\n{brief}\n\nMarket research:\n{research_context}"))


@tool
def legal_agent(brief: str) -> str:
    """Review legal and compliance risks: regulatory exposure, data privacy, IP, due diligence flags.

    Args:
        brief: The investment brief to review
    """
    worker = Agent(system_prompt=LEGAL_PROMPT, callback_handler=None)
    return str(worker(brief))


# The writer is an Agent turned into a tool with .as_tool(). delegate=True makes its memo
# the orchestrator's final answer: no extra orchestrator model call rewrites it, and the
# memo streams once through the orchestrator. preserve_context=False (the default) resets
# the writer before every call, like the fresh workers above.
writer = Agent(
    name="writer_agent",
    system_prompt=WRITER_PROMPT,
    callback_handler=None,
)
writer_agent = writer.as_tool(
    description=(
        "Write the final investment memo from the brief, research findings, financial analysis, "
        "and legal review. Call this LAST, after research_agent, finance_agent, and legal_agent."
    ),
    delegate=True,
)

---

## Part 3: Build and Run the Orchestrator

The orchestrator is a standard `Agent`, but instead of business tools (like `get_company_data`), its tools are other agents. The LLM decides the routing, the arguments and the order of calls.

The input is Module 2's decision brief plus an investment committee request.

In [ ]:
orchestrator = Agent(
    tools=[research_agent, finance_agent, legal_agent, writer_agent],
    system_prompt=ORCHESTRATOR_PROMPT,
)

INVESTMENT_BRIEF = DECISION_BRIEF + """
Investment committee request: produce an investment analysis covering market research,
financial viability, legal risks, and a final recommendation memo (Invest / Invest with conditions / Pass).
"""

print("Running orchestrator (LLM delegates to Research, Finance, Legal, Writer)...")
t0 = time.time()
result = orchestrator(INVESTMENT_BRIEF)
elapsed = time.time() - t0
print(f"\nDone in {elapsed:.1f}s  |  stop_reason: {result.stop_reason}")

---

## Part 4: Inspect What the Orchestrator Decided

In Module 3 the order came from graph edges and showed up in `execution_order`. Here each specialist still runs in its own context, but the routing lives in the orchestrator's `agent.messages`. See which tools it called, in what order, and with what arguments.

In [ ]:
print("=== ORCHESTRATOR TOOL CALLS ===")
call_count = 0
for msg in orchestrator.messages:
    for block in msg["content"]:
        if "toolUse" in block:
            tu = block["toolUse"]
            call_count += 1
            inp = json.dumps(tu["input"])
            print(f"  {call_count}. {tu['name']}({inp[:80]}...)")
print()
print(f"Total tool calls: {call_count}")
print("The order above was chosen by the LLM from the prompt and tool descriptions, not by Python code.")

In [ ]:
# Orchestrator token usage. Each specialist is a separate Agent with its own context;
# its tokens are not included here, only the time its tool call took.
summary = result.metrics.get_summary()
usage = summary["accumulated_usage"]

print(f"{'Metric':<20} {'Value':>10}")
print("-" * 32)
print(f"{'Input tokens':<20} {usage['inputTokens']:>10,}")
print(f"{'Output tokens':<20} {usage['outputTokens']:>10,}")
print(f"{'Total tokens':<20} {usage['totalTokens']:>10,}")
print(f"{'LLM cycles':<20} {summary['total_cycles']:>10}")
print()

print("Per-tool stats:")
for name, data in summary["tool_usage"].items():
    s = data["execution_stats"]
    print(f"  {name}: calls={s['call_count']} | avg_time={s['average_time']:.1f}s")

---

## Key Takeaways

| Concept | What you saw |
|---------|-------------|
| `@tool` wrapping `Agent` | Multi-parameter specialist; the docstring is the routing signal |
| `Agent.as_tool(delegate=True)` | The writer's memo is the final answer, with no extra orchestrator call |
| Focused contexts | Each specialist sees only what the orchestrator passes it |
| LLM routing | No edges wire the specialists: the orchestrator decides order and arguments (Module 3: your edges did) |
| `callback_handler=None` | Silent specialists; the memo streams once through the orchestrator |
| `result.metrics.get_summary()["tool_usage"]` | Call count and timing per specialist |

---

## What's Next

The core workshop ends here. You've run one decision brief through a single agent (Module 2), a fixed graph (Module 3), and an orchestrator that picks its own route (Module 4).

Optional patterns and the capstone are in [`samples/extra/`](../extra/): parallel fork-join, critic-refiner cycles, dynamic swarms, and a capstone that combines them.

---

## Want a real multi-turn conversation?

```bash
cd samples/04-agent-as-tool
pip install -r requirements.txt
python chat.py
```